In [ ]:
"""
*****************************************************************************
* Kevin Randolph
* DAT 340: Client/Server Development
* Professor Tad Kellogg
* Assignment 7-2: Project Two
* 2/22/26
* 
* This file implements a fully interactive MongoDB dashboard for Grazioso
* Salvare. The application connects to the Austin Animal Center Outcomes
* dataset using a custom CRUD Python module and allows users to filter
* dogs by rescue type. The dashboard dynamically updates a data table,
* breed breakdown chart and geolocation map based on user selections.
*
* The application follows an MVC-style structure where MongoDB serves
* as the model, Dash components provide the view and callback functions
* act as the controller to manage user interaction and data updates.
*****************************************************************************
"""

# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Dashboard components
import dash_leaflet as dl
from dash import dcc, html
from dash import dash_table
from dash.dependencies import Input, Output

import plotly.express as px
import pandas as pd
import base64
import os

JupyterDash.infer_jupyter_proxy_config()

# ---------------------------------------
# Import CRUD module 
# ---------------------------------------
from CRUD_Python_Module import AnimalShelter

# ---------------------------------------
# Database connection 
# ---------------------------------------
username = "aacuser"
password = "KRSNHU"
shelter = AnimalShelter(username, password)

# ---------------------------------------
# Load Grazioso Salvare logo
# ---------------------------------------
image_filename = image_filename = "Grazioso Salvare Logo.png"

encoded_logo = base64.b64encode(
    open(image_filename, "rb").read()
).decode("utf-8")

# ---------------------------------------
# Filter queries/controller logic
# ---------------------------------------
def build_query(filter_type):
    if filter_type == "RESET":
        return {}

    if filter_type == "WATER":
        return {
            "animal_type": "Dog",
            "breed": {"$in": ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"]},
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }

    if filter_type == "MOUNTAIN":
        return {
            "animal_type": "Dog",
            "breed": {"$in": ["German Shepherd", "Alaskan Malamute", "Old English Sheepdog", "Siberian Husky", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }

    if filter_type == "DISASTER":
        return {
            "animal_type": "Dog",
            "breed": {"$in": ["Doberman Pinscher", "German Shepherd", "Golden Retriever", "Bloodhound", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}
        }

    # Fallback
    return {"animal_type": "Dog"}


def fetch_dataframe(filter_type):
    query = build_query(filter_type)
    records = shelter.read(query)

    df = pd.DataFrame.from_records(records)

    # Handle empty result sets
    if df.empty:
        return df

    # Drop _id, ObjectId breaks DataTable
    if "_id" in df.columns:
        df.drop(columns=["_id"], inplace=True)

    return df


# Initial load: RESET view
df_initial = fetch_dataframe("RESET")

# ---------------------------------------
# App layout/view
# ---------------------------------------
app = JupyterDash(__name__)

app.layout = html.Div([
    html.Div(id="hidden-div", style={"display": "none"}),

    # Header row with logo/identifier
    html.Div(
        style={"display": "flex", "alignItems": "center", "gap": "20px"},
        children=[
            html.Div(
                children=[
                    html.H1("Grazioso Salvare Dashboard"),
                    html.H3("Unique Identifier: Kevin Randolph")
                ]
            ),
            html.Div(
                children=[
                    html.Img(
                        src=f"data:image/png;base64,{encoded_logo}" if encoded_logo else "",
                        style={"height": "90px"}  # adjust if you want
                    ) if encoded_logo else html.Div("Logo file not found. Check the path: code_files/Grazioso Salvare Logo.png")
                ]
            ),
        ]
    ),

    html.Hr(),

    # Filter controls
    html.Div(
        children=[
            html.H3("Rescue Type Filter"),
            dcc.RadioItems(
                id="filter-type",
                options=[
                    {"label": "Reset", "value": "RESET"},
                    {"label": "Water Rescue", "value": "WATER"},
                    {"label": "Mountain or Wilderness Rescue", "value": "MOUNTAIN"},
                    {"label": "Disaster or Individual Tracking", "value": "DISASTER"},
                ],
                value="RESET",
                labelStyle={"display": "block"}
            ),
        ]
    ),

    html.Hr(),

    # Data table
    dash_table.DataTable(
        id="datatable-id",
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df_initial.columns],
        data=df_initial.to_dict("records"),

        row_selectable="single",
        selected_rows=[0],

        page_size=10,
        sort_action="native",
        filter_action="native",
        column_selectable="single",

        style_table={"overflowX": "auto"},
        style_cell={"textAlign": "left", "minWidth": "120px", "width": "120px", "maxWidth": "220px"},
        style_header={"fontWeight": "bold"},
    ),

    html.Br(),
    html.Hr(),

    # Chart + Map 
    html.Div(
        className="row",
        style={"display": "flex", "gap": "20px"},
        children=[
            html.Div(id="graph-id", className="col s12 m6"),
            html.Div(id="map-id", className="col s12 m6"),
        ]
    )
])


# ---------------------------------------
# Callbacks/controller
# ---------------------------------------

# Update table based on filter selection
@app.callback(
    Output("datatable-id", "data"),
    Output("datatable-id", "columns"),
    Output("datatable-id", "selected_rows"),
    Input("filter-type", "value")
)
def update_dashboard(filter_type):
    df = fetch_dataframe(filter_type)

    # If no rows, return empty table
    if df.empty:
        return [], [], []

    columns = [{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns]
    data = df.to_dict("records")

    # Default select the first row so map always has something to show
    return data, columns, [0]


# Breed pie chart from table data 
@app.callback(
    Output("graph-id", "children"),
    Input("datatable-id", "derived_virtual_data")
)
def update_graphs(viewData):
    if viewData is None or len(viewData) == 0:
        return [html.Div("No data available for chart.")]

    dff = pd.DataFrame.from_dict(viewData)

    if "breed" not in dff.columns:
        return [html.Div("Column 'breed' not found for chart.")]

    fig = px.pie(dff, names="breed", title="Breed Breakdown (Filtered Results)")
    return [dcc.Graph(figure=fig)]


# Highlight selected columns
@app.callback(
    Output("datatable-id", "style_data_conditional"),
    Input("datatable-id", "selected_columns")
)
def update_styles(selected_columns):
    if not selected_columns:
        return []
    return [{
        "if": {"column_id": i},
        "backgroundColor": "#D2F3FF"
    } for i in selected_columns]


# Map updates from selected row
@app.callback(
    Output("map-id", "children"),
    Input("datatable-id", "derived_virtual_data"),
    Input("datatable-id", "derived_virtual_selected_rows")
)
def update_map(viewData, index):
    if viewData is None or len(viewData) == 0:
        return [html.Div("No map data to display for this filter.")]

    dff = pd.DataFrame.from_dict(viewData)

    row = 0
    if index is not None and len(index) > 0:
        row = index[0]
    if row >= len(dff):
        row = 0

    try:
        lat = dff.iloc[row, 13]
        lon = dff.iloc[row, 14]
        breed = dff.iloc[row, 4]
        name = dff.iloc[row, 9]
    except Exception:
        lat = dff["location_lat"].iloc[row] if "location_lat" in dff.columns else 30.75
        lon = dff["location_long"].iloc[row] if "location_long" in dff.columns else -97.48
        breed = dff["breed"].iloc[row] if "breed" in dff.columns else ""
        name = dff["name"].iloc[row] if "name" in dff.columns else ""

    # convert to floats (safe fallback)
    try:
        lat = float(lat)
        lon = float(lon)
    except (TypeError, ValueError):
        lat, lon = 30.75, -97.48

    return [
        dl.Map(
            style={"width": "1000px", "height": "500px"},
            center=[30.75, -97.48],
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),
                dl.Marker(
                    position=[lat, lon],
                    children=[
                        dl.Tooltip(str(breed)),
                        dl.Popup([
                            html.H1("Animal Name"),
                            html.P(str(name))
                        ])
                    ]
                )
            ]
        )
    ]


# Runs app in Codio JupyterLab mode
app.run_server(mode="jupyterlab", host="0.0.0.0", port=8051, debug=False)